In [1]:
!pip install groq -q

In [9]:
!pip install streamlit -q

In [25]:
import os
import re
import pandas as pd
from groq import Groq
from google.colab import userdata

# Configure paths and constants
DATA_PATH = "nutrition_dataset.csv"
NUTRIENTS = {
    "Calories": ("calories_kcal_per_100g", "kcal"),
    "Protein": ("protein_g_per_100g", "g"),
    "Carbohydrates": ("carbs_g_per_100g", "g"),
    "Fat": ("fat_g_per_100g", "g"),
    "Fiber": ("fiber_g_per_100g", "g"),
    "Sugar": ("sugar_g_per_100g", "g"),
}

# Load Data
try:
    # Attempt to load, creating a sample DataFrame if file is not found
    if os.path.exists(DATA_PATH):
        df = pd.read_csv(DATA_PATH)
    else:
        # Fallback sample dataset in case the file doesn't exist yet
        data = {
            "food_name": ["Banana", "Apple", "Chicken Breast", "Broccoli"],
            "calories_kcal_per_100g": [89, 52, 165, 34],
            "protein_g_per_100g": [1.1, 0.3, 31.0, 2.8],
            "carbs_g_per_100g": [22.8, 13.8, 0.0, 7.0],
            "fat_g_per_100g": [0.3, 0.2, 3.6, 0.4],
            "fiber_g_per_100g": [2.6, 2.4, 0.0, 2.6],
            "sugar_g_per_100g": [12.2, 10.4, 0.0, 1.7]
        }
        df = pd.DataFrame(data)
        df.to_csv(DATA_PATH, index=False)
        print(f"Created a sample nutritional dataset at '{DATA_PATH}'")
except Exception as exc:
    print(f"Could not load or create dataset: {exc}")
    df = pd.DataFrame()

def find_food(message):
    """Find a food name mentioned in the user's message (longest exact name first)."""
    text = message.lower()
    foods = sorted(df["food_name"].astype(str).tolist(), key=len, reverse=True)
    for food in foods:
        if re.search(r"(?<![a-z])" + re.escape(food.lower()) + r"(?![a-z])", text):
            return food
    return None

def find_quantity(message):
    """Read a quantity such as 150g, 150 g, or 150 grams."""
    match = re.search(r"(\d+(?:\.\d+)?)\s*(?:g|gram|grams)\b", message.lower())
    return float(match.group(1)) if match else None

def nutrition_result(food, grams):
    row = df[df["food_name"].str.lower() == food.lower()].iloc[0]
    result = {}
    for label, (column, unit) in NUTRIENTS.items():
        result[label] = (float(row[column]) * grams / 100, unit)
    return result

def format_report(food, grams, result):
    lines = [f"### Nutrition for {grams:g} g of {food}", "",
             "Calculated from the dataset:"]
    for label, (value, unit) in result.items():
        lines.append(f"- **{label}:** {value:.2f} {unit}")
    lines.append("")
    lines.append("_Values are approximate and depend on food variety and preparation._")
    return "\n".join(lines)

def ask_groq(user_message, api_key):
    client = Groq(api_key=api_key)
    dataset_context = df.to_csv(index=False)
    system_message = {
        "role": "system",
        "content": (
            "You are a friendly nutrition education assistant. Be concise and clear. "
            "Use ONLY the dataset below for exact numeric food nutrition values. "
            "Values are per 100 grams. Never invent missing food values or claim this small "
            "sample dataset contains every food. If the user asks for a quantity, explain "
            "that the system can calculate it when they provide grams. You may explain general "
            "nutrition concepts, but do not diagnose, prescribe diets, or replace a registered "
            "dietitian or clinician."
            "\n\nDATASET (CSV):\n" + dataset_context
        ),
    }
    response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[system_message, {"role": "user", "content": user_message}],
        temperature=0.3,
        max_tokens=700,
    )
    return response.choices[0].message.content

def analyze_message(user_text):
    print(f"User: {user_text}\n" + "-"*40)
    food = find_food(user_text)
    grams = find_quantity(user_text)

    if food and grams is not None and grams > 0:
        result = nutrition_result(food, grams)
        answer = format_report(food, grams, result)
        print(answer)
    elif food and (grams is None or grams <= 0):
        print(f"I found **{food}** in the dataset. Please specify the quantity in grams (e.g., '150g {food}').")
    else:
        # Access GROQ_API_KEY from environment variables or Colab secrets
        api_key = os.getenv("GROQ_API_KEY") or globals().get('GROQ_API_KEY')
        if not api_key:
            print("Groq API key not found. Please provide an active GROQ_API_KEY variable.")
        else:
            try:
                answer = ask_groq(user_text, api_key)
                print(answer)
            except Exception as exc:
                print(f"Sorry, the Groq request failed: {exc}")

# --- Demo Execution ---
# Let's test the assistant directly with a sample query!
analyze_message("How much nutrition is in 200g Banana?")

User: How much nutrition is in 200g Banana?
----------------------------------------
### Nutrition for 200 g of Banana

Calculated from the dataset:
- **Calories:** 178.00 kcal
- **Protein:** 2.20 g
- **Carbohydrates:** 45.60 g
- **Fat:** 0.60 g
- **Fiber:** 5.20 g
- **Sugar:** 24.40 g

_Values are approximate and depend on food variety and preparation._
